# Step 1 — Data Ingestion & Compliance Classification

**TrialWatch | APAN5400 Data Engineering | Group 8 | Spring 2026**

## What this notebook does
This notebook implements **Step 1** of the TrialWatch ETL pipeline:

1. Loads the AACT (Aggregate Analysis of ClinicalTrials.gov) flat files from Google Drive
2. Initializes a PySpark session on Google Colab to handle the full 581K trial dataset
3. Joins studies, sponsors, and designs tables to build a unified trial record
4. Filters to **Applicable Clinical Trials (ACTs)** under FDAAA 801 criteria
5. Computes FDAAA compliance status: `COMPLIANT / LATE / MISSING / NOT_DUE_YET`
6. Outputs two CSVs to Google Drive:
   - `trialsclean.csv` — 132,185 filtered ACT trials with metadata
   - `compliancemetrics.csv` — compliance status and days overdue per trial

## Data source
- **ClinicalTrials.gov AACT flat files** (pipe-delimited `.txt`)
- Downloaded from https://aact.ctti-clinicaltrials.org/
- 581,102 total trials → filtered to 132,185 ACTs
- Key tables used: `studies`, `sponsors`, `designs`

## Why PySpark?
pandas would crash loading 581K rows with all AACT columns. PySpark processes
the dataset in parallel across Colab's available cores with no memory errors.


In [ ]:
# Install PySpark in Google Colab
# Required because Colab doesn't include PySpark by default
!pip install -q pyspark


## Cell 2 — Mount Google Drive & Verify AACT Path
Mounts Google Drive so we can read the AACT flat files.
Handles both zipped and pre-extracted versions of the AACT dataset.
Also initializes a PySpark session configured for Colab's single-node environment.

In [ ]:
# Mount Google Drive to access AACT flat files
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import zipfile
from pyspark.sql import SparkSession

# ── PATH CONFIGURATION ────────────────────────────────────────────────────────
# Update this path to match where your AACT flat files are stored in Drive
AACT_PATH = "/content/drive/MyDrive/AACT data flattext"

print("AACT path exists:", os.path.exists(AACT_PATH))

if not os.path.exists(AACT_PATH):
    raise FileNotFoundError(f"Folder not found: {AACT_PATH}")

print("\nContents of AACT folder:")
for f in sorted(os.listdir(AACT_PATH))[:50]:
    print(" -", repr(f))

# ── HANDLE ZIPPED VS EXTRACTED FILES ─────────────────────────────────────────
# AACT downloads come as a single zip. If only zip exists, extract it first.
zip_files = [f for f in os.listdir(AACT_PATH) if f.lower().endswith(".zip")]
txt_files = [f for f in os.listdir(AACT_PATH) if f.lower().endswith(".txt")]

if len(txt_files) == 0 and len(zip_files) > 0:
    # Only zip found — extract it
    zip_path = os.path.join(AACT_PATH, zip_files[0])
    extract_path = os.path.join(AACT_PATH, "unzipped")
    os.makedirs(extract_path, exist_ok=True)

    print("\nZip found, extracting now...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

    AACT_PATH = extract_path  # Update path to extracted folder
    print("Updated AACT_PATH:", AACT_PATH)
elif len(txt_files) > 0:
    print("\nTXT files already present. No unzip needed.")
else:
    print("\nNo zip and no txt files found. Verify the folder manually.")

# ── SPARK SESSION INITIALIZATION ──────────────────────────────────────────────
# local[*] uses all available Colab CPU cores
# shuffle.partitions=8 avoids over-partitioning on a single-node Colab runtime
spark = SparkSession.builder \
    .appName("TrialWatch-AACT-Step1") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print("\nSpark started. Version:", spark.version)


## Cell 3 — Load AACT Tables
Reads the pipe-delimited AACT flat files into PySpark DataFrames.
AACT uses `|` as delimiter and double-quote escaping for text fields.

In [ ]:
# ── HELPER: READ ANY AACT TABLE ───────────────────────────────────────────────
# AACT flat files are pipe-delimited (|) with quoted text fields
# multiLine=False improves performance; AACT records don't span multiple lines
def read_aact_table(table_name):
    file_path = os.path.join(AACT_PATH, f"{table_name}.txt")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}")

    return spark.read \
        .option("header", True) \
        .option("sep", "|") \
        .option("quote", '"') \
        .option("escape", '"') \
        .option("multiLine", False) \
        .csv(file_path)

# ── LOAD PRIMARY TABLE ────────────────────────────────────────────────────────
# studies table is the master trial record — one row per NCT ID
studies_df = read_aact_table("studies")
print("studies rows:", studies_df.count())
print("studies columns:", len(studies_df.columns))
studies_df.printSchema()

# ── LOAD SUPPORTING TABLES ────────────────────────────────────────────────────
# These may not exist in all AACT snapshots — wrapped in try/except
def safe_load(table_name):
    try:
        df = read_aact_table(table_name)
        print(f"Loaded {table_name}: {df.count()} rows")
        return df
    except Exception as e:
        print(f"Could not load {table_name}: {e}")
        return None

sponsors_df = safe_load("sponsors")   # Lead sponsor name and org class (INDUSTRY/NIH/OTHER)
designs_df  = safe_load("designs")    # Study phase (Phase 1/2/3/4)


## Cell 4 — Build Unified Trial Table
Joins studies + sponsors + designs into a single DataFrame.
Handles schema variations across AACT snapshot versions by checking column names dynamically.
Filters to LEAD sponsors only (AACT has multiple sponsor rows per trial).

In [ ]:
from pyspark.sql.functions import (
    col, lit, when, coalesce, to_date, regexp_replace,
    datediff, current_date, expr, upper
)

# ── EXTRACT LEAD SPONSOR ──────────────────────────────────────────────────────
# sponsors table has multiple rows per trial (lead + collaborators)
# We only want the LEAD sponsor to avoid duplicate rows after join
if sponsors_df is not None:
    sponsor_cols = sponsors_df.columns
    sponsor_nct_col   = "nct_id" if "nct_id" in sponsor_cols else None
    sponsor_name_col  = "name" if "name" in sponsor_cols else None
    sponsor_class_col = "agency_class" if "agency_class" in sponsor_cols else None
    sponsor_lead_col  = "lead_or_collaborator" if "lead_or_collaborator" in sponsor_cols else None

    if sponsor_nct_col and sponsor_name_col:
        lead_sponsors_df = sponsors_df
        if sponsor_lead_col:
            # Filter to LEAD only — excludes collaborating organizations
            lead_sponsors_df = lead_sponsors_df.filter(upper(col(sponsor_lead_col)) == "LEAD")

        lead_sponsors_df = lead_sponsors_df.select(
            col(sponsor_nct_col).alias("nctid"),
            col(sponsor_name_col).alias("orgname"),
            col(sponsor_class_col).alias("orgclass") if sponsor_class_col else lit(None).alias("orgclass")
        ).dropDuplicates(["nctid"])  # One row per trial
    else:
        lead_sponsors_df = None
else:
    lead_sponsors_df = None

# ── EXTRACT PHASE DATA ────────────────────────────────────────────────────────
# designs table has the clinical phase (Phase 1/2/3/4)
# Used for ACT filtering — we only keep Phase 2/3/4 trials
if designs_df is not None:
    design_cols = designs_df.columns
    design_nct_col = "nct_id" if "nct_id" in design_cols else None
    phase_col      = "phase" if "phase" in design_cols else None

    if design_nct_col and phase_col:
        phase_df = designs_df.select(
            col(design_nct_col).alias("nctid"),
            col(phase_col).alias("phases")
        ).dropDuplicates(["nctid"])
    else:
        phase_df = None
else:
    phase_df = None

# ── BUILD BASE TRIAL TABLE ────────────────────────────────────────────────────
# Select the 6 key fields from the studies table
# Uses dynamic column detection to handle different AACT snapshot schemas
study_cols = studies_df.columns
nct_col              = "nct_id" if "nct_id" in study_cols else None
study_type_col       = "study_type" if "study_type" in study_cols else None
overall_status_col   = "overall_status" if "overall_status" in study_cols else None
primary_completion_col = "primary_completion_date" if "primary_completion_date" in study_cols else None
results_submitted_col  = "results_first_submitted_date" if "results_first_submitted_date" in study_cols else None
study_phase_col      = "phase" if "phase" in study_cols else None

if not nct_col:
    raise ValueError("nct_id column not found in studies table")

trial_df = studies_df.select(
    col(nct_col).alias("nctid"),
    col(study_type_col).alias("studytype")              if study_type_col       else lit(None).alias("studytype"),
    col(overall_status_col).alias("overallstatus")       if overall_status_col   else lit(None).alias("overallstatus"),
    col(primary_completion_col).alias("primarycompletiondate") if primary_completion_col else lit(None).alias("primarycompletiondate"),
    col(results_submitted_col).alias("resultsfirstsubmitdate") if results_submitted_col  else lit(None).alias("resultsfirstsubmitdate"),
    col(study_phase_col).alias("study_phase")            if study_phase_col      else lit(None).alias("study_phase")
)

# Left-join sponsors (some trials have no lead sponsor record)
trial_df = trial_df.join(lead_sponsors_df, on="nctid", how="left") if lead_sponsors_df else \
           trial_df.withColumn("orgname", lit(None)).withColumn("orgclass", lit(None))

# Left-join phase data; fall back to studies.phase if designs.phase missing
trial_df = trial_df.join(phase_df, on="nctid", how="left") if phase_df else \
           trial_df.withColumn("phases", lit(None))

# Prefer designs.phase over studies.phase (more standardized values)
trial_df = trial_df.withColumn(
    "phases", coalesce(col("phases"), col("study_phase"))
).drop("study_phase")

print("Base trial row count:", trial_df.count())
trial_df.show(5, truncate=False)


## Cell 5 — Date Cleaning, ACT Filter & Compliance Classification

### ACT Filter (FDAAA 801 criteria)
To qualify as an Applicable Clinical Trial under FDAAA 801, a study must be:
- **Interventional** study type
- **Phase 2, 3, or 4** (Phase 1 trials are exempt)
- **Primary completion date present** (needed to compute the 12-month deadline)

### Compliance Logic
| Status | Condition |
|---|---|
| `MISSING` | No results submitted AND deadline has passed |
| `NOT_DUE_YET` | No results submitted AND deadline not yet reached |
| `COMPLIANT` | Results submitted on or before deadline |
| `LATE` | Results submitted but after the deadline |

In [ ]:
# ── DATE CLEANING ─────────────────────────────────────────────────────────────
# AACT sometimes stores partial dates as 'YYYY-MM' without a day
# Append '-01' to make them parseable by to_date()
trial_df = trial_df.withColumn(
    "primarycompletiondate_clean",
    regexp_replace(col("primarycompletiondate"), r"^(\d{4}-\d{2})$", "\\1-01")
)

# Convert string dates to proper DateType columns for date arithmetic
trial_df = trial_df.withColumn("primarydt", to_date(col("primarycompletiondate_clean")))
trial_df = trial_df.withColumn("resultsdt", to_date(col("resultsfirstsubmitdate")))

# ── ACT FILTER (FDAAA 801 CRITERIA) ──────────────────────────────────────────
# Only keep interventional trials in Phase 2/3/4 with a primary completion date
# 581,102 total trials → 132,185 applicable trials after this filter
act_df = trial_df.filter(
    (upper(col("studytype")) == "INTERVENTIONAL") &
    (upper(col("phases")).isin("PHASE2", "PHASE3", "PHASE4")) &
    col("primarydt").isNotNull()   # Must have a completion date to compute deadline
)
print("ACT trial count:", act_df.count())

# ── SPONSOR GROUP CLASSIFICATION ──────────────────────────────────────────────
# Map org_class to 3 simplified sponsor groups for dashboard filtering
act_df = act_df.withColumn(
    "sponsorgroup",
    when(upper(col("orgclass")) == "INDUSTRY", lit("INDUSTRY"))
    .when(upper(col("orgclass")).isin("NIH", "FED"), lit("GOVERNMENT"))
    .otherwise(lit("ACADEMICOROTHER"))
)

# ── FDAAA 801 DEADLINE ────────────────────────────────────────────────────────
# Under FDAAA 801, results must be submitted within 365 days of primary completion
act_df = act_df.withColumn("deadline", expr("date_add(primarydt, 365)"))

# ── COMPLIANCE STATUS ─────────────────────────────────────────────────────────
# 4-way classification based on results submission date vs. deadline
act_df = act_df.withColumn(
    "compliancestatusv2",
    when(col("resultsdt").isNull() & (current_date() > col("deadline")),  lit("MISSING"))    # Overdue, never submitted
    .when(col("resultsdt").isNull() & (current_date() <= col("deadline")), lit("NOTDUEYET")) # Deadline not yet reached
    .when(col("resultsdt") <= col("deadline"),                             lit("COMPLIANT")) # Submitted on time
    .otherwise(                                                             lit("LATE"))      # Submitted but late
)

# ── DAYS OVERDUE ─────────────────────────────────────────────────────────────
# Compute how many days past the deadline each trial is
# For MISSING trials: days from deadline to today
# For LATE trials: days from deadline to results submission date
act_df = act_df.withColumn(
    "daysoverdue",
    when(col("compliancestatusv2") == "COMPLIANT",  lit(0))
    .when(col("compliancestatusv2") == "NOTDUEYET", lit(0))
    .when(col("resultsdt").isNull(), datediff(current_date(), col("deadline")))  # MISSING: today - deadline
    .otherwise(datediff(col("resultsdt"), col("deadline")))                      # LATE: submit date - deadline
)

# Boolean flag for easy filtering in MongoDB queries
act_df = act_df.withColumn(
    "isoverdue",
    when(col("compliancestatusv2").isin("LATE", "MISSING"), lit(True)).otherwise(lit(False))
)

print("\nCompliance breakdown:")
act_df.groupBy("compliancestatusv2").count().show()


## Cell 6 — Save Output CSVs to Google Drive
Writes two final CSV files that feed into Step 2 (enrichment) and Step 3 (MongoDB load).
`coalesce(1)` merges Spark's partitioned output into a single CSV file for easier handling.

In [ ]:
import shutil

# ── BUILD FINAL OUTPUT DATAFRAMES ─────────────────────────────────────────────
# trialsclean: trial metadata used by MongoDB 'trials' collection
trialsclean_df = act_df.select(
    "nctid", "orgname", "orgclass", "sponsorgroup", "studytype",
    "phases", "overallstatus", "primarycompletiondate",
    "resultsfirstsubmitdate", "primarydt", "resultsdt"
)

# compliancemetrics: compliance status per trial used by 'compliance_status' collection
compliance_df = act_df.select(
    "nctid", "primarydt", "resultsdt", "deadline",
    "compliancestatusv2", "daysoverdue", "isoverdue"
)

# ── WRITE TO SPARK FOLDERS ────────────────────────────────────────────────────
# Spark writes to a folder with part-00000.csv — we then rename to a single file
# coalesce(1) forces all partitions into one output file
TRIALS_OUT = "/content/drive/MyDrive/trialsclean_spark"
COMP_OUT   = "/content/drive/MyDrive/compliancemetrics_spark"

trialsclean_df.coalesce(1).write.mode("overwrite").option("header", True).csv(TRIALS_OUT)
compliance_df.coalesce(1).write.mode("overwrite").option("header", True).csv(COMP_OUT)

# ── RENAME PART FILE → CLEAN CSV ──────────────────────────────────────────────
# Spark names output 'part-00000-....csv' — find and copy to a clean filename
trial_part = [f for f in os.listdir(TRIALS_OUT) if f.startswith("part-") and f.endswith(".csv")][0]
comp_part  = [f for f in os.listdir(COMP_OUT)   if f.startswith("part-") and f.endswith(".csv")][0]

shutil.copy(os.path.join(TRIALS_OUT, trial_part), "/content/drive/MyDrive/trialsclean.csv")
shutil.copy(os.path.join(COMP_OUT,   comp_part),  "/content/drive/MyDrive/compliancemetrics.csv")

print("Saved: trialsclean.csv")
print("Saved: compliancemetrics.csv")
print("\nStep 1 complete. These files are inputs to Step 2 (enrichment).")

# Store paths for reference in Step 2
TRIALSCSV     = "/content/drive/MyDrive/trialsclean.csv"
COMPLIANCECSV = "/content/drive/MyDrive/compliancemetrics.csv"
